# Classification Model Comparison
## Loan Default Prediction — Logistic Regression vs SVM

**Dataset:** Finance Loan Default Dataset  
**Models:** Logistic Regression, Support Vector Machine (SVM)  
**Goal:** Train and compare two classifiers to predict loan default

## Step 1: Load the Dataset

We load the dataset using pandas. The file uses a .xls extension but is stored as a CSV, so we use read_csv.

In [ ]:
# Import the pandas library for data manipulation
import pandas as pd

# Load the dataset — file uses .xls extension but is stored as a CSV
df = pd.read_csv('finance_loan_default_dataset.xls')

# Preview the first 5 rows to confirm it loaded correctly
print(df.head())

# Check the total number of rows and columns
print(df.shape)

## Step 2: Inspect the Dataset

We inspect the dataset to understand its structure, data types, missing values, and duplicates before making any changes.

In [ ]:
# Display all column names in the dataset
print(df.columns)

# Show data types and non-null counts for each column
print(df.info())

# Display summary statistics for numerical columns
print(df.describe())

# Count missing values in each column
print(df.isnull().sum())

# Check for any duplicate rows
print(df.duplicated().sum())

## Step 3: Clean the Data

We remove duplicate rows and fill missing values.  
Numerical columns are filled with the median. Categorical columns are filled with the most frequent value.

In [ ]:
# Check number of duplicate rows before cleaning
print("Duplicates before:", df.duplicated().sum())

# Remove duplicate rows
df = df.drop_duplicates()

# Confirm duplicates have been removed
print("Duplicates after:", df.duplicated().sum())

# Fill missing numerical values with the median of each column
numerical_columns = ['Age', 'Annual_Income', 'Credit_Score', 'Savings_Balance']
for col in numerical_columns:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with the most frequent value (mode)
categorical_columns = ['Employment_Status', 'Account_Type']
for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Confirm there are no more missing values
print("Missing values after cleaning:")
print(df.isnull().sum())

## Step 4: Define Features and Target

The target variable is what the model predicts — Loan_Default.  
The feature columns are the inputs the model learns from.  
Applicant_ID is a row identifier with no predictive value and is excluded.

In [ ]:
# Define the feature columns the model will learn from
# Applicant_ID is excluded as it is just a row identifier
feature_columns = [
    'Age', 'Annual_Income', 'Credit_Score', 'Loan_Amount',
    'Loan_Term_Months', 'Existing_Debt', 'Late_Payments_Last_Year',
    'Savings_Balance', 'Debt_to_Income_Ratio',
    'Employment_Status', 'Account_Type', 'Has_Credit_Card'
]

# Assign features to X and target to y
X = df[feature_columns]
y = df['Loan_Default']

# Encode the target variable: No -> 0, Yes -> 1
y = y.map({'No': 0, 'Yes': 1})

# Confirm the shapes of X and y
print("X shape:", X.shape)
print("y shape:", y.shape)

# Check the class distribution of the target variable
print(y.value_counts())

## Step 5: Train/Test Split

We split the data into 80% training and 20% testing.  
stratify=y ensures both splits maintain the same class balance.

In [ ]:
# Import the train/test split function
from sklearn.model_selection import train_test_split

# Split the data — 80% for training, 20% for testing
# random_state=42 ensures reproducibility
# stratify=y maintains the same class ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Confirm the size of each split
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## Step 6: Preprocessing

Numerical features are scaled using StandardScaler.  
Categorical features are encoded using OneHotEncoder.  
The preprocessor is fitted on training data only and applied to the test set.

In [ ]:
# Import the necessary preprocessing tools
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define which columns are numerical and which are categorical
numerical_features = [
    'Age', 'Annual_Income', 'Credit_Score', 'Loan_Amount',
    'Loan_Term_Months', 'Existing_Debt', 'Late_Payments_Last_Year',
    'Savings_Balance', 'Debt_to_Income_Ratio'
]

categorical_features = ['Employment_Status', 'Account_Type', 'Has_Credit_Card']

# Build the preprocessor:
# - StandardScaler normalises numerical features to the same scale
# - OneHotEncoder converts categorical features into binary columns
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# Fit the preprocessor on training data only, then apply to both sets
# This prevents data leakage from the test set
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Confirm the new shapes after preprocessing
print("X_train after preprocessing:", X_train_preprocessed.shape)
print("X_test after preprocessing:", X_test_preprocessed.shape)

## Step 7: Train and Evaluate — Logistic Regression

We train a Logistic Regression model and evaluate it using accuracy, precision, recall, F1 score, and a confusion matrix.

In [ ]:
# Import Logistic Regression and all evaluation metrics
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Initialise and train the Logistic Regression model
# max_iter=1000 ensures the model has enough iterations to converge
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_preprocessed, y_train)

# Use the trained model to make predictions on the test set
y_pred_lr = lr_model.predict(X_test_preprocessed)

# Print individual evaluation metrics
print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1 Score:", f1_score(y_test, y_pred_lr))

# Print the full classification report with per-class metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['No Default', 'Default']))

# Plot the confusion matrix to visualise prediction results
cm_lr = confusion_matrix(y_test, y_pred_lr)
ConfusionMatrixDisplay(cm_lr, display_labels=['No Default', 'Default']).plot()
plt.title("Confusion Matrix — Logistic Regression")
plt.show()

## Step 8: Train and Evaluate — SVM

We train a Support Vector Machine model and evaluate it using the same metrics as Logistic Regression.

In [ ]:
# Import the Support Vector Machine classifier
from sklearn.svm import SVC

# Initialise and train the SVM model with an RBF kernel
# RBF (Radial Basis Function) is a good general-purpose kernel
svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train_preprocessed, y_train)

# Use the trained model to make predictions on the test set
y_pred_svm = svm_model.predict(X_test_preprocessed)

# Print individual evaluation metrics
print("=== SVM ===")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Precision:", precision_score(y_test, y_pred_svm))
print("Recall:", recall_score(y_test, y_pred_svm))
print("F1 Score:", f1_score(y_test, y_pred_svm))

# Print the full classification report with per-class metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm, target_names=['No Default', 'Default']))

# Plot the confusion matrix to visualise prediction results
cm_svm = confusion_matrix(y_test, y_pred_svm)
ConfusionMatrixDisplay(cm_svm, display_labels=['No Default', 'Default']).plot()
plt.title("Confusion Matrix — SVM")
plt.show()

## Step 9: Compare Both Models

In [ ]:
# Build a comparison dictionary with metrics from both models
comparison = {
    'Model': ['Logistic Regression', 'SVM'],

    # Overall accuracy of each model on the test set
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_svm)
    ],

    # Precision — of all predicted defaults, how many actually defaulted
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_svm)
    ],

    # Recall — of all actual defaults, how many did the model catch
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_svm)
    ],

    # F1 Score — harmonic mean of precision and recall
    'F1 Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_svm)
    ]
}

# Convert to a DataFrame for a clean side-by-side view
comparison_df = pd.DataFrame(comparison)
comparison_df = comparison_df.set_index('Model')

# Display the results rounded to 3 decimal places
print(comparison_df.round(3))

## Task 2: Results Interpretation

### Which model performed better?
SVM outperformed Logistic Regression across all metrics. SVM achieved 87% accuracy compared to Logistic Regression's 80%, and also achieved a higher recall on the Default class, meaning it caught more actual defaults. SVM is the stronger model for this problem.

### Which metric is most important for this business problem?
Recall is the most important metric. In loan default prediction, the cost of missing an actual default (false negative) is much higher than the cost of a false alarm (false positive). A bank that misses a defaulting applicant faces direct financial loss, so the model must prioritise catching as many true defaults as possible.

### What do false positives and false negatives mean in this dataset?
- **False positive:** The model predicts a loan applicant will default, but they would not have. The bank may unfairly reject a creditworthy applicant.
- **False negative:** The model predicts an applicant will not default, but they do. The bank approves a risky loan and may suffer financial loss. This is the more costly error.

### What is one possible limitation or bias in the model?
The dataset is imbalanced — roughly 75% of applicants did not default and only 25% did. Both models are biased towards predicting No Default because they saw far more examples of that class during training. This means the models may underperform on the minority class (Default) in real-world conditions.

### Why should human judgment still be used?
Machine learning models learn from historical data, which may reflect past biases in lending decisions. A model cannot account for context outside the data — such as a sudden job loss, economic changes, or individual circumstances. Human review ensures that edge cases are handled fairly, that decisions can be explained to applicants, and that the bank remains accountable for its lending practices.